<a href="https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Task type: Classification

I am framing my lane as a binary classification problem. The goal is to predict whether a content page is declining or not. Each page can be assigned to one of two classes: declining or not declining. This classification can support prioritizing content pages for refresh.

In [25]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

task_type = "classification"

print("Task type:", task_type)
print("Dataset shape:", df.shape)

Task type: classification
Dataset shape: (30000, 44)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target: is_declining

I would predict whether a content page is declining. I define the target as `is_declining = 1` when the observed `trend_direction` is "down", and `0` otherwise. This target comes from an observed trend field in the starter data, converted into a binary label for classification.

In [26]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

df[["content_id", "trend_direction", "is_declining"]].head(10)

,content_id,trend_direction,is_declining
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric: Precision@K

I would use Precision@K as the main success metric because the content team has limited capacity to refresh pages. The goal is to make the top-ranked recommendations useful. A higher Precision@K means that a larger share of the selected pages are actually declining. This metric therefore measures whether the model supports effective content-refresh prioritization.

In [27]:
# Define the evaluation metric for content-refresh prioritization
metric = "Precision@K"
print("Success metric:", metric)

Success metric: Precision@K


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of analysis: one content page

The unit of analysis is one content page. Each row represents one content item and contains its observed performance, engagement, trend, and other available signals. The prediction would therefore be made at the content-page level.

In [28]:
import os

print(os.listdir("/content/flyrank-ml-internship"))
print(os.path.exists("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"))

['AGENTS.md', 'CLAUDE.md', '.gitignore', 'data', 'SETUP.md', 'outputs', 'LICENSE', 'scripts', 'requirements.txt', '.git', 'DATA_USE.md', 'README.md', 'skills', '.github', 'notebooks', 'work', 'GUIDE.md', 'docs', 'submission']
True


In [29]:
# Show the unit of analysis: one row represents one content page
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [30]:
# Sketch the classification target
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

df[["content_id", "trend_direction", "is_declining"]].head(10)

,content_id,trend_direction,is_declining
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


In [31]:
print("Target distribution:")
print(df["is_declining"].value_counts())

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML instead of a fixed rule?

A fixed rule such as "refresh every page with low CTR" uses one manually chosen threshold and can miss the context of a page. Content performance is influenced by multiple signals, including impressions, clicks, sessions, CTR, engagement, content age, and position. ML can learn patterns across these signals and estimate which pages are more likely to be declining. This makes the output useful for ranking and prioritizing pages for content refresh rather than applying the same rule to every page.

In [32]:
# Show some of the signals available for the ML task
signals = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "content_age_days",
    "avg_position"
]

df[signals].head()

,impressions_90d,clicks_90d,sessions_90d,ctr,engagement_rate,content_age_days,avg_position
0,3803,29,17,0.76,5.88,187,10.6
1,15320,7,9,0.05,0.00,445,20.3
2,12581,11,11,0.09,0.00,141,36.5
3,11751,58,78,0.49,1.28,463,6.2
4,19140,24,145,0.13,0.00,263,44.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.